# 068 — Síntesis de voz y clonación responsable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Pipeline TTS:** texto → **normalización** (números, siglas, fechas) + **G2P** (letras →
fonemas) → **modelo acústico** (Tacotron 2: seq2seq con atención que predice el log-mel de
80 bandas, hop ~12,5 ms, más un *stop token*) → **vocoder** (mel → onda, porque el mel
descarta la fase).

**Vocoders:** WaveNet (2016) genera muestra a muestra con convoluciones causales dilatadas
(`p(x) = ∏ p(xₜ|x₁…xₜ₋₁)`): calidad casi humana, pero 16 000 pasos por segundo de audio.
Los vocoders paralelos (HiFi-GAN) generan toda la onda en una pasada → tiempo real.

**Clonación:** un *speaker encoder* comprime segundos de voz en un d-vector que condiciona
el modelo acústico (SV2TTS). **Responsabilidad:** consentimiento explícito y documentado,
watermarking imperceptible (degradable por re-grabación/compresión) y detección de audio
sintético. **Evaluación:** MOS (1-5, subjetivo, reportar media *y* desvío) y WER de un ASR
sobre el audio sintético como proxy de inteligibilidad.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Campo receptivo de WaveNet.** Una pila tiene 10 convoluciones causales con
kernel 2 y dilaciones 1, 2, 4, …, 512. (a) Calcula el campo receptivo de **2 pilas
apiladas** en muestras y en milisegundos a 16 kHz. (b) ¿Cuántos pasos secuenciales exige
generar 1,5 s de audio a 16 kHz con un vocoder autoregresivo? ¿Y con uno paralelo?

**Ejercicio 2 — Dimensiones del modelo acústico.** Una frase sintetizada dura 4 s y el mel
usa hop de 12,5 ms con 80 bandas. ¿Cuántas tramas predice el modelo acústico y qué forma
tiene la matriz de salida?

**Ejercicio 3 — MOS con la misma media.** Sistema A: [4, 4, 5, 3, 4]; sistema B:
[5, 5, 5, 1, 4]. Calcula media y desvío estándar (poblacional) de cada uno. ¿Cuál
elegirías para un lector de noticias y por qué la media sola no decide?

**Ejercicio 4 — En código.** Implementa `receptive_field(dilaciones, pilas)` y
`mos(ratings)` (media y desvío) y verifica los ejercicios 1 y 3.


In [ ]:
# TODO: ejecuta run_lab("generation", seed=68)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ
# Ejercicio 4: campo receptivo y MOS
def receptive_field(dilaciones, pilas=1):
    # campo = 1 + pilas * suma(dilaciones)
    return None

def mos(ratings):
    # devuelve (media, desvío poblacional)
    return None

dil = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
# imprime receptive_field(dil, 2), mos([4,4,5,3,4]), mos([5,5,5,1,4])


## Reflexión

1. Una familia pide clonar la voz de un pariente fallecido para un homenaje. ¿Quién puede
   consentir en ese caso, y qué límites de alcance y revocación pondrías por escrito?
2. El watermark de tu TTS sobrevive a la compresión MP3 pero no a la re-grabación con un
   micrófono. ¿Sigue siendo útil? ¿Dentro de qué estrategia de defensa más amplia?
3. Tu TTS lee recetas médicas en voz alta (clase 072). ¿Qué error del frontend de
   normalización sería el más peligroso ("500 mg", "c/8 h") y cómo lo detectarías antes de
   desplegar?
